# 00 - Concepts Quickstart

Purpose: build pre-paper intuition with lightweight hands-on checks.

Estimated time: 45-90 minutes

## What you'll cover

1. Residual stream shape intuition
2. Attention and MLP as additive writers
3. Logit-difference metric
4. Minimal causal check via ablation

In [ ]:
import random
import numpy as np
import torch
from transformer_lens import HookedTransformer

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## Load Model

In [ ]:
MODEL_NAME = 'gpt2'
model = HookedTransformer.from_pretrained(MODEL_NAME)
prompt = 'When John and Mary went to the store, John gave a bottle to'
tokens = model.to_tokens(prompt)
logits, cache = model.run_with_cache(tokens)
tokens.shape, logits.shape

## 1) Residual Stream Shape Intuition

In [ ]:
resid_post_l5 = cache['blocks.5.hook_resid_post']
print('resid_post_l5 shape:', tuple(resid_post_l5.shape))
print('Interpretation: [batch, position, d_model]')

## 2) Attention + MLP Are Additive Writers

In [ ]:
attn_out_l5 = cache['blocks.5.hook_attn_out']
mlp_out_l5 = cache['blocks.5.hook_mlp_out']
resid_pre_l5 = cache['blocks.5.hook_resid_pre']
resid_post_l5 = cache['blocks.5.hook_resid_post']

reconstruction_error = (resid_post_l5 - (resid_pre_l5 + attn_out_l5 + mlp_out_l5)).abs().mean().item()
print('Mean abs reconstruction error:', reconstruction_error)
print('Near-zero means additive intuition is working for this block.')

## 3) Logit Difference Metric

In [ ]:
# Example continuation choice: ' Mary' vs ' John'
correct_tok = model.to_single_token(' Mary')
foil_tok = model.to_single_token(' John')
final_logits = logits[0, -1]
logit_diff = (final_logits[correct_tok] - final_logits[foil_tok]).item()
print('logit_diff (Mary - John):', logit_diff)

## 4) Minimal Causal Check (Ablation)

This is not a full experiment, just a concept demo.

In [ ]:
def zero_mlp_out(act, hook):
    return torch.zeros_like(act)

ablated_logits = model.run_with_hooks(
    tokens,
    fwd_hooks=[('blocks.5.hook_mlp_out', zero_mlp_out)]
)[0]

clean_ld = (logits[0, -1, correct_tok] - logits[0, -1, foil_tok]).item()
ablated_ld = (ablated_logits[-1, correct_tok] - ablated_logits[-1, foil_tok]).item()
ablation_drop = clean_ld - ablated_ld

print('clean logit_diff:', clean_ld)
print('ablated logit_diff:', ablated_ld)
print('ablation_drop:', ablation_drop)

## Reflection Prompts

- Which evidence above is correlational vs causal?
- What can this ablation *not* prove?
- Which concept still feels fuzzy before paper reading?